
# Tutorial: Fine-Tune a Landmark Emotion Model Aligned to Py-Feat ResMaskNet

Audience:
- Researchers replacing Py-Feat's image-only `resmasknet` emotion classifier input with MediaPipe facial landmarks.

Prerequisites:
- Run [prepare_fer2013_landmarks.ipynb](./prepare_fer2013_landmarks.ipynb) first.
- The generated CSVs exist under `tmp/mediapipe_landmark_emotion/dataset`.
- Use the repo Python 3.10 environment with `torch`, `mediapipe`, `opencv-python`, `pandas`, `scikit-learn`, and `matplotlib` installed.

Learning goals:
- Load the FER2013 landmark dataset generated from MediaPipe FaceMesh.
- Train a landmark-only student model in the same 7-class emotion space as Py-Feat's `resmasknet`.
- Backtest multiple landmark model configurations and save comparable artifacts.
- Run single-image inference where an image is converted to MediaPipe landmarks before prediction.

Important modeling note:
- The original Py-Feat `resmasknet` is a CNN over raw face images. It cannot directly accept a `468 x 3` landmark vector.
- This notebook therefore trains a **landmark student / adapter model** that keeps the ResMaskNet label space, optimizer recipe, and evaluation target while replacing the raw-image input with MediaPipe landmarks.
- An optional teacher-distillation path is included for users who want the landmark student to mimic Py-Feat `resmasknet` probabilities in addition to the FER2013 hard labels.



## Outline

1. Load paths, constants, and experiment configurations.
2. Define all helper functions for preprocessing, training, evaluation, artifact saving, and inference.
3. Load the landmark CSVs produced by the FER2013 preparation notebook.
4. Backtest multiple landmark model configurations on the official validation split.
5. Retrain the best configuration on `train + validation` with a small internal holdout, then evaluate on `test`.
6. Run inference from a single image by extracting MediaPipe landmarks, standardizing them with the saved checkpoint, and predicting emotion probabilities.


In [ ]:

from __future__ import annotations

import copy
import json
import os
import random
import time
from pathlib import Path
from typing import Any, Optional

import cv2
import matplotlib.pyplot as plt
import mediapipe as mp
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler, StandardScaler
from torch import nn
from torch.nn import functional as F
from torch.optim import AdamW, RAdam
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.utils.data import DataLoader, TensorDataset

plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.max_colwidth', 120)

def find_repo_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / 'emotional_expressivity').exists() and (candidate / 'openwillis-face').exists():
            return candidate
    raise FileNotFoundError('Could not locate the repository root from the current working directory.')

REPO_ROOT = find_repo_root(Path.cwd())
NOTEBOOK_DIR = REPO_ROOT / 'emotional_expressivity' / 'mediapipe_landmark_emotion'
DATASET_ID = 'AutumnQiu/fer2013'
DATA_DIR = REPO_ROOT / 'tmp' / 'mediapipe_landmark_emotion' / 'dataset'
RUNS_DIR = REPO_ROOT / 'tmp' / 'mediapipe_landmark_emotion'
CONFIG_DIR = NOTEBOOK_DIR / 'configs'
RESULTS_TSV = RUNS_DIR / 'results.tsv'
UPSCALE_SIZE = 224
LANDMARK_COUNT = 468
FEATURE_DIM = LANDMARK_COUNT * 3
EMOTION_COLUMNS = ['anger', 'disgust', 'fear', 'happiness', 'sadness', 'surprise', 'neutral']
LANDMARK_COLUMNS = [
    f'lmk{index:03d}_{axis}'
    for index in range(LANDMARK_COUNT)
    for axis in ('x', 'y', 'z')
]

RUNS_DIR.mkdir(parents=True, exist_ok=True)

pd.DataFrame([
    {
        'repo_root': str(REPO_ROOT),
        'dataset_dir': str(DATA_DIR),
        'runs_dir': str(RUNS_DIR),
        'results_tsv': str(RESULTS_TSV),
        'feature_dim': FEATURE_DIM,
        'num_classes': len(EMOTION_COLUMNS),
    }
])


In [ ]:

BASE_CONFIG = {
    'seed': 1706,
    'device': 'auto',
    'model_type': 'resmask_landmark',
    'num_classes': len(EMOTION_COLUMNS),
    'feature_dim': FEATURE_DIM,
    'hidden_dims': [2048, 1024, 512, 256],
    'dropout': 0.20,
    'batch_size': 64,
    'lr': 1.5e-4,
    'weight_decay': 5e-4,
    'max_epochs': 50,
    'scheduler_factor': 0.5,
    'scheduler_patience': 3,
    'early_stopping_patience': 8,
    'label_smoothing': 0.02,
    'scaler': 'robust',
    'feature_transform': 'signed_log1p',
    'class_weighting': 'balanced',
    'norm_layer': 'layernorm',
    'selection_metric': 'macro_f1',
    'validation_strategy': 'official',
    'final_validation_fraction': 0.05,
    'use_teacher_distillation': False,
    'teacher_cache_path': None,
    'distillation_alpha': 0.20,
    'distillation_temperature': 2.0,
    'num_workers': 0,
    'pin_memory': False,
    'max_train_rows': None,
    'max_validation_rows': None,
    'max_test_rows': None,
}

BACKTEST_EXPERIMENTS = [
    {
        **BASE_CONFIG,
        'experiment_tag': 'resmask_recipe_mlp',
        'notes': 'Closest landmark analogue to the published ResMaskNet optimizer recipe.',
        'model_type': 'mlp',
        'hidden_dims': [512, 256],
        'dropout': 0.30,
        'lr': 1.0e-4,
        'weight_decay': 1.0e-3,
        'output_subdir': 'training_resmask_recipe_mlp',
    },
    {
        **BASE_CONFIG,
        'experiment_tag': 'resmask_landmark_plus',
        'notes': 'Residual MLP adapter that usually performs better on landmark-only FER2013 inputs.',
        'output_subdir': 'training_resmask_landmark_plus',
    },
]

OPTIONAL_TEACHER_DISTILLATION_EXPERIMENT = {
    **BASE_CONFIG,
    'experiment_tag': 'resmask_landmark_distilled',
    'notes': 'Optional landmark student with Py-Feat ResMaskNet teacher distillation.',
    'use_teacher_distillation': True,
    'teacher_cache_path': str(RUNS_DIR / 'teacher_cache_train_validation.npz'),
    'output_subdir': 'training_resmask_landmark_distilled',
}

existing_json_configs = []
for config_path in sorted(CONFIG_DIR.glob('*.json')):
    payload = json.loads(config_path.read_text())
    payload['config_file'] = config_path.name
    existing_json_configs.append(payload)

pd.DataFrame(existing_json_configs)


In [ ]:


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def resolve_device(device_name: str) -> torch.device:
    requested = (device_name or 'auto').lower()
    if requested == 'auto':
        if torch.cuda.is_available():
            return torch.device('cuda')
        if hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
            return torch.device('mps')
        return torch.device('cpu')
    if requested == 'cuda' and torch.cuda.is_available():
        return torch.device('cuda')
    if requested == 'mps' and hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
        return torch.device('mps')
    return torch.device('cpu')


def load_split_dataframe(csv_path: Path, max_rows: Optional[int] = None) -> pd.DataFrame:
    df = pd.read_csv(csv_path)
    if max_rows is not None:
        df = df.iloc[:max_rows].copy()
    return df.reset_index(drop=True)


def apply_feature_transform(values: np.ndarray, mode: str) -> np.ndarray:
    array = np.asarray(values, dtype=np.float32)
    if mode in (None, 'none'):
        return array.astype(np.float32)
    if mode in ('signed_log1p', 'log_geom'):
        return np.sign(array) * np.log1p(np.abs(array))
    raise ValueError(f'Unsupported feature_transform={mode!r}')


def build_scaler(name: str):
    scaler_name = (name or 'standard').lower()
    if scaler_name == 'standard':
        return StandardScaler()
    if scaler_name == 'robust':
        return RobustScaler(with_centering=True, with_scaling=True, quantile_range=(10.0, 90.0))
    raise ValueError(f'Unsupported scaler={name!r}')


def scaler_center_and_scale(scaler) -> tuple[np.ndarray, np.ndarray]:
    if hasattr(scaler, 'mean_'):
        center = np.asarray(scaler.mean_, dtype=np.float32)
        scale = np.asarray(scaler.scale_, dtype=np.float32)
    else:
        center = np.asarray(scaler.center_, dtype=np.float32)
        scale = np.asarray(scaler.scale_, dtype=np.float32)
    scale = np.where(np.abs(scale) < 1e-6, 1.0, scale).astype(np.float32)
    return center, scale


def compute_class_weights(labels: np.ndarray, num_classes: int) -> np.ndarray:
    counts = np.bincount(labels.astype(np.int64), minlength=num_classes).astype(np.float32)
    weights = counts.sum() / np.maximum(counts, 1.0)
    return (weights / weights.mean()).astype(np.float32)


def normalize_landmark_vector(landmark_vector: np.ndarray) -> np.ndarray:
    coords = np.asarray(landmark_vector, dtype=np.float32).reshape(LANDMARK_COUNT, 3)
    coords = coords - coords.mean(axis=0, keepdims=True)
    spans = coords.max(axis=0) - coords.min(axis=0)
    scale = float(np.max(spans))
    if scale < 1e-6:
        scale = 1.0
    return (coords / scale).reshape(-1)


def rgb_from_dataset_image(image: Any, upscale_size: int = UPSCALE_SIZE) -> np.ndarray:
    pil_image = image.convert('RGB') if hasattr(image, 'convert') else image
    image_rgb = np.array(pil_image, dtype=np.uint8)
    if image_rgb.shape[:2] != (upscale_size, upscale_size):
        image_rgb = cv2.resize(image_rgb, (upscale_size, upscale_size), interpolation=cv2.INTER_CUBIC)
    return image_rgb


def make_norm(dim: int, norm_layer: str) -> nn.Module:
    normalized = (norm_layer or 'layernorm').lower()
    if normalized == 'layernorm':
        return nn.LayerNorm(dim)
    if normalized == 'batchnorm':
        return nn.BatchNorm1d(dim)
    raise ValueError(f'Unsupported norm_layer={norm_layer!r}')


class PlainLandmarkMLP(nn.Module):
    def __init__(self, input_dim: int, hidden_dims: list[int], dropout: float, num_classes: int, norm_layer: str) -> None:
        super().__init__()
        if not hidden_dims:
            raise ValueError('hidden_dims must not be empty for PlainLandmarkMLP.')
        layers: list[nn.Module] = []
        previous_dim = input_dim
        for hidden_dim in hidden_dims:
            layers.extend([
                nn.Linear(previous_dim, hidden_dim),
                make_norm(hidden_dim, norm_layer),
                nn.GELU(),
                nn.Dropout(dropout),
            ])
            previous_dim = hidden_dim
        layers.append(nn.Linear(previous_dim, num_classes))
        self.network = nn.Sequential(*layers)

    def forward(self, features: torch.Tensor) -> torch.Tensor:
        return self.network(features)


class ResidualFeedForwardBlock(nn.Module):
    def __init__(self, dim: int, dropout: float, norm_layer: str, expansion: int = 2) -> None:
        super().__init__()
        self.norm = make_norm(dim, norm_layer)
        self.ff = nn.Sequential(
            nn.Linear(dim, dim * expansion),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(dim * expansion, dim),
            nn.Dropout(dropout),
        )

    def forward(self, features: torch.Tensor) -> torch.Tensor:
        return features + self.ff(self.norm(features))


class LandmarkResMaskNetAdapter(nn.Module):
    def __init__(self, input_dim: int, hidden_dims: list[int], dropout: float, num_classes: int, norm_layer: str) -> None:
        super().__init__()
        if not hidden_dims:
            raise ValueError('hidden_dims must not be empty for LandmarkResMaskNetAdapter.')
        first_dim = hidden_dims[0]
        self.stem = nn.Sequential(
            nn.Linear(input_dim, first_dim),
            make_norm(first_dim, norm_layer),
            nn.GELU(),
            nn.Dropout(dropout),
        )
        self.blocks = nn.ModuleList([ResidualFeedForwardBlock(first_dim, dropout, norm_layer)])
        self.transitions = nn.ModuleList()
        previous_dim = first_dim
        for hidden_dim in hidden_dims[1:]:
            self.transitions.append(nn.Sequential(
                nn.Linear(previous_dim, hidden_dim),
                make_norm(hidden_dim, norm_layer),
                nn.GELU(),
                nn.Dropout(dropout),
            ))
            self.blocks.append(ResidualFeedForwardBlock(hidden_dim, dropout, norm_layer))
            previous_dim = hidden_dim
        self.head = nn.Sequential(
            make_norm(previous_dim, norm_layer),
            nn.Linear(previous_dim, num_classes),
        )

    def forward(self, features: torch.Tensor) -> torch.Tensor:
        x = self.stem(features)
        x = self.blocks[0](x)
        for transition, block in zip(self.transitions, self.blocks[1:]):
            x = transition(x)
            x = block(x)
        return self.head(x)


def build_model(config: dict[str, Any]) -> nn.Module:
    model_type = config['model_type']
    if model_type == 'mlp':
        return PlainLandmarkMLP(
            input_dim=int(config['feature_dim']),
            hidden_dims=list(config['hidden_dims']),
            dropout=float(config['dropout']),
            num_classes=int(config['num_classes']),
            norm_layer=str(config['norm_layer']),
        )
    if model_type == 'resmask_landmark':
        return LandmarkResMaskNetAdapter(
            input_dim=int(config['feature_dim']),
            hidden_dims=list(config['hidden_dims']),
            dropout=float(config['dropout']),
            num_classes=int(config['num_classes']),
            norm_layer=str(config['norm_layer']),
        )
    raise ValueError(f'Unsupported model_type={model_type!r}')


def dataframe_to_arrays(df: pd.DataFrame, feature_transform: str) -> tuple[np.ndarray, np.ndarray, list[str]]:
    features = df[LANDMARK_COLUMNS].to_numpy(dtype=np.float32)
    features = apply_feature_transform(features, feature_transform)
    labels = df['emotion'].to_numpy(dtype=np.int64)
    sample_ids = df['sample_id'].astype(str).tolist()
    return features.astype(np.float32), labels, sample_ids


def make_loader(
    features: np.ndarray,
    labels: np.ndarray,
    batch_size: int,
    shuffle: bool,
    teacher_probs: Optional[np.ndarray] = None,
    num_workers: int = 0,
    pin_memory: bool = False,
) -> DataLoader:
    tensors: list[torch.Tensor] = [
        torch.from_numpy(features.astype(np.float32)),
        torch.from_numpy(labels.astype(np.int64)),
    ]
    if teacher_probs is not None:
        tensors.append(torch.from_numpy(teacher_probs.astype(np.float32)))
    dataset = TensorDataset(*tensors)
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=num_workers,
        pin_memory=pin_memory,
        drop_last=False,
    )


def batch_teacher_probs(batch: tuple[torch.Tensor, ...]) -> Optional[torch.Tensor]:
    return batch[2] if len(batch) > 2 else None


def summarize_predictions(labels: np.ndarray, predictions: np.ndarray, probabilities: np.ndarray) -> dict[str, Any]:
    matrix = confusion_matrix(labels, predictions, labels=np.arange(len(EMOTION_COLUMNS)))
    report = classification_report(
        labels,
        predictions,
        labels=np.arange(len(EMOTION_COLUMNS)),
        target_names=EMOTION_COLUMNS,
        zero_division=0,
        output_dict=True,
    )
    per_class_f1 = {
        emotion: float(report[emotion]['f1-score'])
        for emotion in EMOTION_COLUMNS
    }
    return {
        'accuracy': float(accuracy_score(labels, predictions)),
        'balanced_accuracy': float(balanced_accuracy_score(labels, predictions)),
        'macro_f1': float(f1_score(labels, predictions, average='macro', zero_division=0)),
        'weighted_f1': float(f1_score(labels, predictions, average='weighted', zero_division=0)),
        'per_class_f1': per_class_f1,
        'confusion_matrix': matrix.tolist(),
        'classification_report': report,
        'probabilities_preview': probabilities[:5].tolist(),
    }


def evaluate_model(model: nn.Module, loader: DataLoader, criterion: nn.Module, device: torch.device) -> dict[str, Any]:
    model.eval()
    losses: list[float] = []
    all_logits: list[torch.Tensor] = []
    all_labels: list[torch.Tensor] = []
    with torch.no_grad():
        for batch in loader:
            features = batch[0].to(device)
            labels = batch[1].to(device)
            logits = model(features)
            loss = criterion(logits, labels)
            losses.append(float(loss.item()))
            all_logits.append(logits.detach().cpu())
            all_labels.append(labels.detach().cpu())
    logits = torch.cat(all_logits, dim=0)
    labels = torch.cat(all_labels, dim=0).numpy()
    probabilities = torch.softmax(logits, dim=1).numpy()
    predictions = probabilities.argmax(axis=1)
    summary = summarize_predictions(labels, predictions, probabilities)
    summary['loss'] = float(np.mean(losses)) if losses else 0.0
    summary['labels'] = labels.tolist()
    summary['predictions'] = predictions.tolist()
    return summary


def build_optimizer(model: nn.Module, config: dict[str, Any]):
    optimizer_name = str(config.get('optimizer', 'adamw')).lower()
    if optimizer_name == 'radam':
        return RAdam(model.parameters(), lr=float(config['lr']), weight_decay=float(config['weight_decay']))
    if optimizer_name == 'adamw':
        return AdamW(model.parameters(), lr=float(config['lr']), weight_decay=float(config['weight_decay']))
    raise ValueError(f'Unsupported optimizer={optimizer_name!r}')


def train_model(
    model: nn.Module,
    train_loader: DataLoader,
    holdout_loader: DataLoader,
    criterion: nn.Module,
    optimizer,
    scheduler,
    config: dict[str, Any],
    device: torch.device,
) -> tuple[list[dict[str, Any]], dict[str, Any], dict[str, Any]]:
    selection_metric = str(config['selection_metric'])
    best_score = -np.inf
    best_state: Optional[dict[str, torch.Tensor]] = None
    best_holdout_metrics: Optional[dict[str, Any]] = None
    best_epoch = 0
    history: list[dict[str, Any]] = []
    epochs_without_improvement = 0

    for epoch in range(1, int(config['max_epochs']) + 1):
        model.train()
        train_losses: list[float] = []
        train_ce_losses: list[float] = []
        train_distill_losses: list[float] = []
        start_time = time.time()

        for batch in train_loader:
            features = batch[0].to(device)
            labels = batch[1].to(device)
            teacher_probs = batch_teacher_probs(batch)
            if teacher_probs is not None:
                teacher_probs = teacher_probs.to(device)

            optimizer.zero_grad(set_to_none=True)
            logits = model(features)
            ce_loss = criterion(logits, labels)
            loss = ce_loss
            distill_loss_value = 0.0

            if teacher_probs is not None and float(config.get('distillation_alpha', 0.0)) > 0.0:
                temperature = float(config.get('distillation_temperature', 1.0))
                distill_loss = F.kl_div(
                    F.log_softmax(logits / temperature, dim=1),
                    teacher_probs.clamp_min(1e-8),
                    reduction='batchmean',
                ) * (temperature ** 2)
                alpha = float(config['distillation_alpha'])
                loss = (1.0 - alpha) * ce_loss + alpha * distill_loss
                distill_loss_value = float(distill_loss.item())

            loss.backward()
            optimizer.step()

            train_losses.append(float(loss.item()))
            train_ce_losses.append(float(ce_loss.item()))
            train_distill_losses.append(distill_loss_value)

        holdout_metrics = evaluate_model(model, holdout_loader, criterion, device)
        scheduler.step(holdout_metrics['loss'])
        epoch_seconds = time.time() - start_time

        history_row = {
            'epoch': epoch,
            'train_loss': float(np.mean(train_losses)) if train_losses else 0.0,
            'train_ce_loss': float(np.mean(train_ce_losses)) if train_ce_losses else 0.0,
            'train_distill_loss': float(np.mean(train_distill_losses)) if train_distill_losses else 0.0,
            'holdout_loss': float(holdout_metrics['loss']),
            'holdout_accuracy': float(holdout_metrics['accuracy']),
            'holdout_balanced_accuracy': float(holdout_metrics['balanced_accuracy']),
            'holdout_macro_f1': float(holdout_metrics['macro_f1']),
            'holdout_weighted_f1': float(holdout_metrics['weighted_f1']),
            'learning_rate': float(optimizer.param_groups[0]['lr']),
            'epoch_seconds': float(epoch_seconds),
        }
        history.append(history_row)

        score = float(holdout_metrics[selection_metric])
        print(
            f"epoch={epoch:02d} "
            f"train_loss={history_row['train_loss']:.4f} "
            f"holdout_loss={history_row['holdout_loss']:.4f} "
            f"holdout_macro_f1={history_row['holdout_macro_f1']:.4f} "
            f"holdout_balanced_accuracy={history_row['holdout_balanced_accuracy']:.4f} "
            f"lr={history_row['learning_rate']:.6f}"
        )

        if score > best_score + 1e-6:
            best_score = score
            best_epoch = epoch
            best_state = {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}
            best_holdout_metrics = copy.deepcopy(holdout_metrics)
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1

        if epochs_without_improvement >= int(config['early_stopping_patience']):
            print(f'Early stopping at epoch={epoch}.')
            break

    if best_state is None or best_holdout_metrics is None:
        raise RuntimeError('Training completed without producing a best checkpoint.')

    model.load_state_dict(best_state)
    metadata = {
        'best_epoch': int(best_epoch),
        'best_score': float(best_score),
        'selection_metric': selection_metric,
    }
    return history, best_holdout_metrics, metadata


def save_json(payload: dict[str, Any], path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open('w', encoding='utf-8') as handle:
        json.dump(payload, handle, indent=2)
        handle.write('\n')


def render_history_plot(history_df: pd.DataFrame, title: str = 'Training history') -> None:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(history_df['epoch'], history_df['train_loss'], label='train_loss')
    axes[0].plot(history_df['epoch'], history_df['holdout_loss'], label='holdout_loss')
    axes[0].set_title(title)
    axes[0].set_xlabel('epoch')
    axes[0].set_ylabel('loss')
    axes[0].legend()

    axes[1].plot(history_df['epoch'], history_df['holdout_macro_f1'], label='holdout_macro_f1')
    axes[1].plot(history_df['epoch'], history_df['holdout_balanced_accuracy'], label='holdout_balanced_accuracy')
    axes[1].plot(history_df['epoch'], history_df['holdout_accuracy'], label='holdout_accuracy')
    axes[1].set_title('Holdout metrics')
    axes[1].set_xlabel('epoch')
    axes[1].set_ylabel('score')
    axes[1].legend()
    plt.tight_layout()
    plt.show()


def render_confusion_matrix(matrix: list[list[int]] | np.ndarray, title: str) -> None:
    array = np.asarray(matrix, dtype=np.int64)
    fig, ax = plt.subplots(figsize=(7, 6))
    image = ax.imshow(array, cmap='Blues')
    ax.set_xticks(np.arange(len(EMOTION_COLUMNS)))
    ax.set_yticks(np.arange(len(EMOTION_COLUMNS)))
    ax.set_xticklabels(EMOTION_COLUMNS, rotation=45, ha='right')
    ax.set_yticklabels(EMOTION_COLUMNS)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')
    ax.set_title(title)
    for row_index in range(array.shape[0]):
        for col_index in range(array.shape[1]):
            ax.text(col_index, row_index, int(array[row_index, col_index]), ha='center', va='center', color='black')
    fig.colorbar(image, ax=ax, fraction=0.046, pad=0.04)
    plt.tight_layout()
    plt.show()


def save_run_plots(history_df: pd.DataFrame, holdout_metrics: dict[str, Any], output_dir: Path) -> None:
    plots_dir = output_dir / 'plots'
    plots_dir.mkdir(parents=True, exist_ok=True)

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(history_df['epoch'], history_df['train_loss'], label='train_loss')
    axes[0].plot(history_df['epoch'], history_df['holdout_loss'], label='holdout_loss')
    axes[0].set_title('Training history')
    axes[0].set_xlabel('epoch')
    axes[0].set_ylabel('loss')
    axes[0].legend()

    axes[1].plot(history_df['epoch'], history_df['holdout_macro_f1'], label='holdout_macro_f1')
    axes[1].plot(history_df['epoch'], history_df['holdout_balanced_accuracy'], label='holdout_balanced_accuracy')
    axes[1].plot(history_df['epoch'], history_df['holdout_accuracy'], label='holdout_accuracy')
    axes[1].set_title('Holdout metrics')
    axes[1].set_xlabel('epoch')
    axes[1].set_ylabel('score')
    axes[1].legend()
    plt.tight_layout()
    fig.savefig(plots_dir / 'training_history.png', dpi=160, bbox_inches='tight')
    plt.close(fig)

    matrix = np.asarray(holdout_metrics['confusion_matrix'], dtype=np.int64)
    fig, ax = plt.subplots(figsize=(7, 6))
    image = ax.imshow(matrix, cmap='Blues')
    ax.set_xticks(np.arange(len(EMOTION_COLUMNS)))
    ax.set_yticks(np.arange(len(EMOTION_COLUMNS)))
    ax.set_xticklabels(EMOTION_COLUMNS, rotation=45, ha='right')
    ax.set_yticklabels(EMOTION_COLUMNS)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')
    ax.set_title('Holdout confusion matrix')
    for row_index in range(matrix.shape[0]):
        for col_index in range(matrix.shape[1]):
            ax.text(col_index, row_index, int(matrix[row_index, col_index]), ha='center', va='center', color='black')
    fig.colorbar(image, ax=ax, fraction=0.046, pad=0.04)
    plt.tight_layout()
    fig.savefig(plots_dir / 'confusion_matrix.png', dpi=160, bbox_inches='tight')
    plt.close(fig)


def append_results_row(row: dict[str, Any], results_tsv_path: Path) -> None:
    frame = pd.DataFrame([row])
    if results_tsv_path.exists():
        existing = pd.read_csv(results_tsv_path, sep='	')
        combined = pd.concat([existing, frame], ignore_index=True)
    else:
        combined = frame
    combined.to_csv(results_tsv_path, sep='	', index=False)


def maybe_load_teacher_probs(sample_ids: list[str], cache_path: Optional[str]) -> Optional[np.ndarray]:
    if not cache_path:
        return None
    path = Path(cache_path)
    if not path.exists():
        raise FileNotFoundError(
            f'Teacher cache not found: {path}. Build it with build_resmasknet_teacher_cache(...) first.'
        )
    payload = np.load(path, allow_pickle=True)
    mapping = {
        str(sample_id): probabilities
        for sample_id, probabilities in zip(payload['sample_ids'].tolist(), payload['probabilities'])
    }
    missing = [sample_id for sample_id in sample_ids if sample_id not in mapping]
    if missing:
        raise KeyError(f'{len(missing)} sample_ids are missing from the teacher cache. Example: {missing[:3]}')
    return np.asarray([mapping[sample_id] for sample_id in sample_ids], dtype=np.float32)


def parse_sample_id(sample_id: str) -> tuple[str, int]:
    split_name, index_str = sample_id.rsplit('_', 1)
    return split_name, int(index_str)


def build_resmasknet_teacher_cache(
    sample_ids: list[str],
    cache_path: Path,
    hf_token: Optional[str] = None,
    force: bool = False,
) -> Path:
    if cache_path.exists() and not force:
        print(f'Reusing existing teacher cache: {cache_path}')
        return cache_path

    from datasets import load_dataset

    os.environ.setdefault('OMP_NUM_THREADS', '1')
    import feat

    dataset = load_dataset(DATASET_ID, token=hf_token)
    detector = feat.Detector()

    split_mapping = {'validation': 'valid' if 'valid' in dataset else 'validation'}
    cached_sample_ids: list[str] = []
    cached_probabilities: list[np.ndarray] = []
    fallback_count = 0

    for sample_id in sample_ids:
        split_name, row_index = parse_sample_id(sample_id)
        hf_split = split_mapping.get(split_name, split_name)
        example = dataset[hf_split][row_index]
        image_rgb = rgb_from_dataset_image(example['image'])
        image_bgr = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2BGR)

        try:
            faces = detector.detect_faces(image_bgr, threshold=0.5)
            landmarks = detector.detect_landmarks(image_bgr, detected_faces=faces)
            emotions = detector.detect_emotions(image_bgr, faces, landmarks)
            probabilities = np.asarray(emotions[0][0], dtype=np.float32)
        except Exception:
            probabilities = np.zeros(len(EMOTION_COLUMNS), dtype=np.float32)
            probabilities[int(example['label'])] = 1.0
            fallback_count += 1

        cached_sample_ids.append(sample_id)
        cached_probabilities.append(probabilities)

    cache_path.parent.mkdir(parents=True, exist_ok=True)
    np.savez_compressed(
        cache_path,
        sample_ids=np.asarray(cached_sample_ids, dtype=str),
        probabilities=np.asarray(cached_probabilities, dtype=np.float32),
    )
    print(f'Wrote teacher cache to {cache_path} with fallback_count={fallback_count}.')
    return cache_path


def choose_fit_and_holdout_frames(
    train_df: pd.DataFrame,
    validation_df: pd.DataFrame,
    config: dict[str, Any],
    final_mode: bool,
) -> tuple[pd.DataFrame, pd.DataFrame, str]:
    if final_mode:
        combined = pd.concat([train_df, validation_df], ignore_index=True)
        fit_df, holdout_df = train_test_split(
            combined,
            test_size=float(config['final_validation_fraction']),
            random_state=int(config['seed']),
            stratify=combined['emotion'],
        )
        return fit_df.reset_index(drop=True), holdout_df.reset_index(drop=True), 'internal_holdout'

    if str(config['validation_strategy']) != 'official':
        combined = pd.concat([train_df, validation_df], ignore_index=True)
        fit_df, holdout_df = train_test_split(
            combined,
            test_size=float(config['final_validation_fraction']),
            random_state=int(config['seed']),
            stratify=combined['emotion'],
        )
        return fit_df.reset_index(drop=True), holdout_df.reset_index(drop=True), 'internal_holdout'

    return train_df.copy().reset_index(drop=True), validation_df.copy().reset_index(drop=True), 'validation'


def run_experiment(
    config: dict[str, Any],
    train_df: pd.DataFrame,
    validation_df: pd.DataFrame,
    test_df: pd.DataFrame,
    final_mode: bool = False,
) -> dict[str, Any]:
    experiment = copy.deepcopy(config)
    set_seed(int(experiment['seed']))
    device = resolve_device(str(experiment['device']))
    experiment['resolved_device'] = str(device)
    experiment.setdefault('optimizer', 'adamw')

    fit_df, holdout_df, evaluation_split = choose_fit_and_holdout_frames(train_df, validation_df, experiment, final_mode=final_mode)
    if experiment.get('max_train_rows') is not None:
        fit_df = fit_df.iloc[: int(experiment['max_train_rows'])].copy()
    if experiment.get('max_validation_rows') is not None:
        holdout_df = holdout_df.iloc[: int(experiment['max_validation_rows'])].copy()
    test_eval_df = test_df.copy().reset_index(drop=True)
    if experiment.get('max_test_rows') is not None:
        test_eval_df = test_eval_df.iloc[: int(experiment['max_test_rows'])].copy()

    fit_features_raw, fit_labels, fit_sample_ids = dataframe_to_arrays(fit_df, str(experiment['feature_transform']))
    holdout_features_raw, holdout_labels, holdout_sample_ids = dataframe_to_arrays(holdout_df, str(experiment['feature_transform']))
    test_features_raw, test_labels, test_sample_ids = dataframe_to_arrays(test_eval_df, str(experiment['feature_transform']))

    scaler = build_scaler(str(experiment['scaler']))
    fit_features = scaler.fit_transform(fit_features_raw).astype(np.float32)
    holdout_features = scaler.transform(holdout_features_raw).astype(np.float32)
    test_features = scaler.transform(test_features_raw).astype(np.float32)
    scaler_center, scaler_scale = scaler_center_and_scale(scaler)

    teacher_probs = None
    if bool(experiment.get('use_teacher_distillation')):
        teacher_probs = maybe_load_teacher_probs(fit_sample_ids, experiment.get('teacher_cache_path'))

    pin_memory = bool(experiment.get('pin_memory', False)) and device.type == 'cuda'
    train_loader = make_loader(
        fit_features,
        fit_labels,
        batch_size=int(experiment['batch_size']),
        shuffle=True,
        teacher_probs=teacher_probs,
        num_workers=int(experiment.get('num_workers', 0)),
        pin_memory=pin_memory,
    )
    holdout_loader = make_loader(
        holdout_features,
        holdout_labels,
        batch_size=int(experiment['batch_size']),
        shuffle=False,
        num_workers=int(experiment.get('num_workers', 0)),
        pin_memory=pin_memory,
    )
    test_loader = make_loader(
        test_features,
        test_labels,
        batch_size=int(experiment['batch_size']),
        shuffle=False,
        num_workers=int(experiment.get('num_workers', 0)),
        pin_memory=pin_memory,
    )

    model = build_model(experiment).to(device)
    class_weights = None
    if str(experiment.get('class_weighting', 'none')) == 'balanced':
        class_weights = torch.from_numpy(compute_class_weights(fit_labels, int(experiment['num_classes']))).to(device)

    criterion = nn.CrossEntropyLoss(
        weight=class_weights,
        label_smoothing=float(experiment.get('label_smoothing', 0.0)),
    )
    optimizer = build_optimizer(model, experiment)
    scheduler = ReduceLROnPlateau(
        optimizer,
        mode='min',
        factor=float(experiment['scheduler_factor']),
        patience=int(experiment['scheduler_patience']),
    )

    history, holdout_metrics, metadata = train_model(
        model=model,
        train_loader=train_loader,
        holdout_loader=holdout_loader,
        criterion=criterion,
        optimizer=optimizer,
        scheduler=scheduler,
        config=experiment,
        device=device,
    )
    test_metrics = evaluate_model(model, test_loader, criterion, device)

    output_subdir = experiment.get('output_subdir') or f"training_{experiment['experiment_tag']}"
    if final_mode:
        output_subdir = f'{output_subdir}_final'
    output_dir = RUNS_DIR / output_subdir
    output_dir.mkdir(parents=True, exist_ok=True)

    history_df = pd.DataFrame(history)
    history_path = output_dir / 'training_history.csv'
    history_df.to_csv(history_path, index=False)

    classification_report_path = output_dir / 'classification_report.json'
    confusion_matrix_path = output_dir / 'confusion_matrix.csv'
    pd.DataFrame(
        holdout_metrics['confusion_matrix'],
        index=EMOTION_COLUMNS,
        columns=EMOTION_COLUMNS,
    ).to_csv(confusion_matrix_path)
    save_json(holdout_metrics['classification_report'], classification_report_path)
    save_run_plots(history_df, holdout_metrics, output_dir)

    checkpoint = {
        'model_state_dict': model.state_dict(),
        'config': experiment,
        'emotion_columns': EMOTION_COLUMNS,
        'landmark_columns': LANDMARK_COLUMNS,
        'scaler_name': str(experiment['scaler']),
        'scaler_center': scaler_center.tolist(),
        'scaler_scale': scaler_scale.tolist(),
        'feature_transform': str(experiment['feature_transform']),
        'best_epoch': int(metadata['best_epoch']),
        'selection_metric': str(metadata['selection_metric']),
        'holdout_metrics': holdout_metrics,
        'test_metrics': test_metrics,
    }
    best_model_path = output_dir / 'best_model.pt'
    torch.save(checkpoint, best_model_path)

    metrics_payload = {
        'mode': 'final' if final_mode else 'experiment',
        'config': experiment,
        'best_epoch': int(metadata['best_epoch']),
        'best_score': float(metadata['best_score']),
        'selection_metric': str(metadata['selection_metric']),
        'evaluation_split': evaluation_split,
        'fit_rows': int(len(fit_df)),
        'holdout_rows': int(len(holdout_df)),
        'test_rows': int(len(test_eval_df)),
        'holdout_metrics': holdout_metrics,
        'test_metrics': test_metrics,
        'output_dir': str(output_dir),
        'best_model_path': str(best_model_path),
        'training_history_path': str(history_path),
        'classification_report_path': str(classification_report_path),
        'confusion_matrix_csv_path': str(confusion_matrix_path),
    }
    metrics_path = output_dir / 'metrics.json'
    save_json(metrics_payload, metrics_path)

    summary_row = {
        'timestamp_utc': pd.Timestamp.utcnow().isoformat(),
        'mode': metrics_payload['mode'],
        'experiment_tag': experiment['experiment_tag'],
        'notes': experiment.get('notes', ''),
        'seed': experiment['seed'],
        'resolved_device': str(device),
        'model_type': experiment['model_type'],
        'hidden_dims': json.dumps(experiment['hidden_dims']),
        'dropout': experiment['dropout'],
        'batch_size': experiment['batch_size'],
        'lr': experiment['lr'],
        'weight_decay': experiment['weight_decay'],
        'max_epochs': experiment['max_epochs'],
        'early_stopping_patience': experiment['early_stopping_patience'],
        'scaler': experiment['scaler'],
        'feature_transform': experiment['feature_transform'],
        'class_weighting': experiment['class_weighting'],
        'optimizer': experiment['optimizer'],
        'norm_layer': experiment['norm_layer'],
        'label_smoothing': experiment['label_smoothing'],
        'use_teacher_distillation': experiment['use_teacher_distillation'],
        'best_epoch': metrics_payload['best_epoch'],
        'best_score': metrics_payload['best_score'],
        'selection_metric': metrics_payload['selection_metric'],
        'evaluation_split': metrics_payload['evaluation_split'],
        'holdout_loss': holdout_metrics['loss'],
        'holdout_accuracy': holdout_metrics['accuracy'],
        'holdout_balanced_accuracy': holdout_metrics['balanced_accuracy'],
        'holdout_macro_f1': holdout_metrics['macro_f1'],
        'holdout_weighted_f1': holdout_metrics['weighted_f1'],
        'test_loss': test_metrics['loss'],
        'test_accuracy': test_metrics['accuracy'],
        'test_balanced_accuracy': test_metrics['balanced_accuracy'],
        'test_macro_f1': test_metrics['macro_f1'],
        'test_weighted_f1': test_metrics['weighted_f1'],
        'output_dir': str(output_dir),
        'best_model_path': str(best_model_path),
        'metrics_json_path': str(metrics_path),
    }
    append_results_row(summary_row, RESULTS_TSV)

    return {
        'config': experiment,
        'output_dir': output_dir,
        'best_model_path': best_model_path,
        'metrics_path': metrics_path,
        'history_path': history_path,
        'history_df': history_df,
        'holdout_metrics': holdout_metrics,
        'test_metrics': test_metrics,
        'evaluation_split': evaluation_split,
        'fit_rows': len(fit_df),
        'holdout_rows': len(holdout_df),
        'test_rows': len(test_eval_df),
    }


def build_leaderboard(run_outputs: list[dict[str, Any]]) -> pd.DataFrame:
    rows = []
    for run in run_outputs:
        rows.append({
            'experiment_tag': run['config']['experiment_tag'],
            'model_type': run['config']['model_type'],
            'holdout_split': run['evaluation_split'],
            'holdout_macro_f1': run['holdout_metrics']['macro_f1'],
            'holdout_balanced_accuracy': run['holdout_metrics']['balanced_accuracy'],
            'holdout_accuracy': run['holdout_metrics']['accuracy'],
            'test_macro_f1': run['test_metrics']['macro_f1'],
            'test_balanced_accuracy': run['test_metrics']['balanced_accuracy'],
            'test_accuracy': run['test_metrics']['accuracy'],
            'output_dir': str(run['output_dir']),
        })
    return pd.DataFrame(rows).sort_values(by=['holdout_macro_f1', 'holdout_balanced_accuracy'], ascending=False).reset_index(drop=True)


def extract_landmarks_and_mesh(image_rgb: np.ndarray):
    with mp.solutions.face_mesh.FaceMesh(
        static_image_mode=True,
        max_num_faces=1,
        refine_landmarks=False,
        min_detection_confidence=0.5,
    ) as face_mesh:
        results = face_mesh.process(image_rgb)
        if not results.multi_face_landmarks:
            return None, None
        face_landmarks = results.multi_face_landmarks[0]
        coords = np.array(
            [[point.x, point.y, point.z] for point in face_landmarks.landmark[:LANDMARK_COUNT]],
            dtype=np.float32,
        )
        return coords.reshape(-1), face_landmarks


def draw_landmarks_overlay(image_rgb: np.ndarray, face_landmarks: Any) -> np.ndarray:
    annotated = image_rgb.copy()
    mp.solutions.drawing_utils.draw_landmarks(
        image=annotated,
        landmark_list=face_landmarks,
        connections=mp.solutions.face_mesh.FACEMESH_TESSELATION,
        landmark_drawing_spec=None,
        connection_drawing_spec=mp.solutions.drawing_styles.get_default_face_mesh_tesselation_style(),
    )
    return annotated


def checkpoint_feature_vector(landmark_vector: np.ndarray, checkpoint: dict[str, Any]) -> np.ndarray:
    normalized_vector = normalize_landmark_vector(landmark_vector)
    transformed = apply_feature_transform(normalized_vector, str(checkpoint['feature_transform']))
    center = np.asarray(checkpoint['scaler_center'], dtype=np.float32)
    scale = np.asarray(checkpoint['scaler_scale'], dtype=np.float32)
    scale = np.where(np.abs(scale) < 1e-6, 1.0, scale)
    standardized = (np.asarray(transformed, dtype=np.float32) - center) / scale
    return standardized.astype(np.float32)


def load_checkpoint_model(checkpoint_path: Path, device_name: Optional[str] = None):
    checkpoint = torch.load(checkpoint_path, map_location='cpu')
    runtime_device = resolve_device(device_name or checkpoint['config'].get('device', 'auto'))
    model = build_model(checkpoint['config']).to(runtime_device)
    model.load_state_dict(checkpoint['model_state_dict'])
    model.eval()
    return checkpoint, model, runtime_device


def run_single_image_inference(image_path: str | Path, checkpoint_path: str | Path):
    checkpoint, inference_model, inference_device = load_checkpoint_model(Path(checkpoint_path))
    image_bgr = cv2.imread(str(image_path))
    if image_bgr is None:
        raise FileNotFoundError(f'Image not found: {image_path}')
    image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
    landmark_vector, face_landmarks = extract_landmarks_and_mesh(image_rgb)
    if landmark_vector is None or face_landmarks is None:
        raise RuntimeError('MediaPipe did not detect a face in the provided image.')

    feature_vector = checkpoint_feature_vector(landmark_vector, checkpoint)
    feature_tensor = torch.from_numpy(feature_vector).unsqueeze(0).to(inference_device)
    with torch.no_grad():
        logits = inference_model(feature_tensor)
        probabilities = torch.softmax(logits, dim=1).cpu().numpy()[0]
        predicted_index = int(torch.argmax(logits, dim=1).item())

    result_row = {
        'image_path': str(image_path),
        'predicted_emotion': EMOTION_COLUMNS[predicted_index],
        'feature_shape': tuple(feature_tensor.shape),
    }
    result_row.update({emotion: float(probabilities[index]) for index, emotion in enumerate(EMOTION_COLUMNS)})
    return {
        'checkpoint': checkpoint,
        'model': inference_model,
        'device': inference_device,
        'image_rgb': image_rgb,
        'landmark_vector': landmark_vector,
        'face_landmarks': face_landmarks,
        'feature_tensor': feature_tensor,
        'probabilities': probabilities,
        'predicted_index': predicted_index,
        'result_frame': pd.DataFrame([result_row]),
    }


In [ ]:

train_df = load_split_dataframe(DATA_DIR / 'train.csv', max_rows=BASE_CONFIG['max_train_rows'])
validation_df = load_split_dataframe(DATA_DIR / 'validation.csv', max_rows=BASE_CONFIG['max_validation_rows'])
test_df = load_split_dataframe(DATA_DIR / 'test.csv', max_rows=BASE_CONFIG['max_test_rows'])

dataset_label_lookup = pd.concat(
    [
        train_df[['sample_id', 'emotion', 'emotion_name']],
        validation_df[['sample_id', 'emotion', 'emotion_name']],
        test_df[['sample_id', 'emotion', 'emotion_name']],
    ],
    ignore_index=True,
).drop_duplicates(subset=['sample_id']).set_index('sample_id')

split_overview = pd.DataFrame([
    {'split': 'train', 'rows': len(train_df), 'class_balance': train_df['emotion_name'].value_counts().to_dict()},
    {'split': 'validation', 'rows': len(validation_df), 'class_balance': validation_df['emotion_name'].value_counts().to_dict()},
    {'split': 'test', 'rows': len(test_df), 'class_balance': test_df['emotion_name'].value_counts().to_dict()},
])

split_overview



## Optional: build a Py-Feat ResMaskNet teacher cache

Run the next cell only if you want teacher distillation. It reloads FER2013 from Hugging Face, runs Py-Feat's image-based detector, and stores teacher probabilities keyed by `sample_id`.

For normal landmark-only supervised training, skip this section.


In [ ]:

# Example for optional teacher distillation.
# Uncomment and run when you want the landmark student to mimic Py-Feat ResMaskNet outputs.
#
# from dotenv import load_dotenv
# load_dotenv(REPO_ROOT / '.env')
# hf_token = os.getenv('HF_TOKEN')
# teacher_sample_ids = pd.concat([train_df['sample_id'], validation_df['sample_id']], ignore_index=True).astype(str).tolist()
# build_resmasknet_teacher_cache(
#     sample_ids=teacher_sample_ids,
#     cache_path=Path(OPTIONAL_TEACHER_DISTILLATION_EXPERIMENT['teacher_cache_path']),
#     hf_token=hf_token,
#     force=False,
# )



## Backtest candidate landmark models

This block trains each experiment configuration against the official validation split and writes a comparable artifact bundle for every run:
- `best_model.pt`
- `metrics.json`
- `training_history.csv`
- `classification_report.json`
- `confusion_matrix.csv`
- `plots/*.png`


In [ ]:
backtest_runs = []
for experiment_config in BACKTEST_EXPERIMENTS:
    print(f"\n=== Running backtest: {experiment_config['experiment_tag']} ===")
    backtest_runs.append(
        run_experiment(
            config=experiment_config,
            train_df=train_df,
            validation_df=validation_df,
            test_df=test_df,
            final_mode=False,
        )
    )

leaderboard = build_leaderboard(backtest_runs)
leaderboard


In [ ]:

if leaderboard.empty:
    raise RuntimeError('No backtest runs were produced.')

best_backtest_run = next(
    run for run in backtest_runs
    if run['config']['experiment_tag'] == leaderboard.iloc[0]['experiment_tag']
)

print('Selected best backtest configuration:')
pd.DataFrame([
    {
        'experiment_tag': best_backtest_run['config']['experiment_tag'],
        'model_type': best_backtest_run['config']['model_type'],
        'holdout_macro_f1': best_backtest_run['holdout_metrics']['macro_f1'],
        'holdout_balanced_accuracy': best_backtest_run['holdout_metrics']['balanced_accuracy'],
        'test_macro_f1': best_backtest_run['test_metrics']['macro_f1'],
        'output_dir': str(best_backtest_run['output_dir']),
    }
])



## Final training run

The final run reuses the best backtest configuration, fits on `train + validation`, keeps a small internal holdout for early stopping, and reports the untouched official `test` split as the final evaluation.


In [ ]:
final_config = copy.deepcopy(best_backtest_run['config'])
final_config['experiment_tag'] = f"{best_backtest_run['config']['experiment_tag']}_final"
final_config['notes'] = f"Final train+validation fit derived from {best_backtest_run['config']['experiment_tag']}"

print(f"\n=== Running final fit: {final_config['experiment_tag']} ===")
final_run = run_experiment(
    config=final_config,
    train_df=train_df,
    validation_df=validation_df,
    test_df=test_df,
    final_mode=True,
)

final_summary = pd.DataFrame([
    {
        'experiment_tag': final_config['experiment_tag'],
        'holdout_split': final_run['evaluation_split'],
        'fit_rows': final_run['fit_rows'],
        'holdout_rows': final_run['holdout_rows'],
        'test_rows': final_run['test_rows'],
        'holdout_macro_f1': final_run['holdout_metrics']['macro_f1'],
        'holdout_balanced_accuracy': final_run['holdout_metrics']['balanced_accuracy'],
        'test_macro_f1': final_run['test_metrics']['macro_f1'],
        'test_balanced_accuracy': final_run['test_metrics']['balanced_accuracy'],
        'test_accuracy': final_run['test_metrics']['accuracy'],
        'best_model_path': str(final_run['best_model_path']),
        'metrics_path': str(final_run['metrics_path']),
    }
])
final_summary


In [ ]:

render_history_plot(final_run['history_df'], title=f"Training history: {final_config['experiment_tag']}")
render_confusion_matrix(final_run['test_metrics']['confusion_matrix'], title='Official test confusion matrix')

pd.DataFrame([
    {
        'split': 'internal_holdout',
        'loss': final_run['holdout_metrics']['loss'],
        'accuracy': final_run['holdout_metrics']['accuracy'],
        'balanced_accuracy': final_run['holdout_metrics']['balanced_accuracy'],
        'macro_f1': final_run['holdout_metrics']['macro_f1'],
        'weighted_f1': final_run['holdout_metrics']['weighted_f1'],
    },
    {
        'split': 'official_test',
        'loss': final_run['test_metrics']['loss'],
        'accuracy': final_run['test_metrics']['accuracy'],
        'balanced_accuracy': final_run['test_metrics']['balanced_accuracy'],
        'macro_f1': final_run['test_metrics']['macro_f1'],
        'weighted_f1': final_run['test_metrics']['weighted_f1'],
    },
])



## Inference visualization

This section uses the trained model exactly as intended:

1. pick an image and display it
2. extract facial landmarks with MediaPipe, then visualize the landmark geometry and the overlay on the original image
3. receive the emotion prediction from those landmarks alone

If the selected image filename maps back to a known dataset `sample_id`, the notebook also shows the original dataset label.
For user-supplied images outside the dataset, it shows only the model outputs.


In [ ]:

best_checkpoint, model, device = load_checkpoint_model(final_run['best_model_path'])

preview_image_path = REPO_ROOT / 'tmp' / 'mediapipe_landmark_emotion' / 'dataset' / 'previews' / 'train_000000.png'
image_bgr = cv2.imread(str(preview_image_path))
if image_bgr is None:
    raise FileNotFoundError(f'Preview image not found: {preview_image_path}')

image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
sample_id = preview_image_path.stem
expected_label_row = dataset_label_lookup.loc[sample_id] if sample_id in dataset_label_lookup.index else None

# Step 1: pick image and display it
plt.figure(figsize=(4, 4))
plt.imshow(image_rgb)
plt.title('Step 1: selected image')
plt.axis('off')
plt.show()

if expected_label_row is not None:
    print(
        f"Dataset label: {expected_label_row['emotion_name']} "
        f"(class_id={int(expected_label_row['emotion'])}, sample_id={sample_id})"
    )
else:
    print(f'No dataset label found for sample_id={sample_id}. Using model-only inference.')

# Step 2: extract landmarks and visualize geometry plus overlay
landmark_vector, face_landmarks = extract_landmarks_and_mesh(image_rgb)
if landmark_vector is None or face_landmarks is None:
    raise RuntimeError('MediaPipe did not detect a face in the preview image.')

landmark_coords = landmark_vector.reshape(LANDMARK_COUNT, 3)
landmark_overlay = draw_landmarks_overlay(image_rgb, face_landmarks)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].scatter(landmark_coords[:, 0], landmark_coords[:, 1], s=8, c=landmark_coords[:, 2], cmap='viridis')
axes[0].invert_yaxis()
axes[0].set_title('Step 2a: landmark geometry')
axes[0].set_xlabel('x')
axes[0].set_ylabel('y')
axes[1].imshow(landmark_overlay)
axes[1].set_title('Step 2b: landmarks on image')
axes[1].axis('off')
plt.tight_layout()
plt.show()

# Step 3: receive prediction from facial landmarks only
normalized_vector = normalize_landmark_vector(landmark_vector)
standardized_vector = checkpoint_feature_vector(landmark_vector, best_checkpoint)
feature_tensor = torch.from_numpy(standardized_vector.astype(np.float32)).unsqueeze(0).to(device)

with torch.no_grad():
    logits = model(feature_tensor)
    probabilities = torch.softmax(logits, dim=1).cpu().numpy()[0]
    predicted_index = int(torch.argmax(logits, dim=1).item())

print(f'Model input feature shape: {tuple(feature_tensor.shape)}')
print(f'Predicted emotion from landmarks: {EMOTION_COLUMNS[predicted_index]}')

result_row = {name: float(probabilities[i]) for i, name in enumerate(EMOTION_COLUMNS)}
if expected_label_row is not None:
    result_row['dataset_emotion'] = str(expected_label_row['emotion_name'])
    result_row['dataset_emotion_id'] = int(expected_label_row['emotion'])
result_row['predicted_emotion'] = EMOTION_COLUMNS[predicted_index]
result_row['sample_id'] = sample_id
pd.DataFrame([result_row])


In [ ]:

# Generic user-facing inference helper.
# Replace the image path below with any image that contains a detectable face.

user_image_path = REPO_ROOT / 'tmp' / 'mediapipe_landmark_emotion' / 'dataset' / 'previews' / 'validation_000000.png'
inference_output = run_single_image_inference(user_image_path, final_run['best_model_path'])

plt.figure(figsize=(4, 4))
plt.imshow(inference_output['image_rgb'])
plt.title('User-supplied image')
plt.axis('off')
plt.show()

plt.figure(figsize=(4, 4))
plt.imshow(draw_landmarks_overlay(inference_output['image_rgb'], inference_output['face_landmarks']))
plt.title('Detected MediaPipe landmarks')
plt.axis('off')
plt.show()

inference_output['result_frame']
